# Notebook 07 — Validation Harness

**Prerequisites:**
- Notebook 04 completed (docs ingested)
- Ollama running with models loaded
- Postgres running

**What to validate:**
- Load gold YAML → canned_questions table
- Run each question as Global Auditor (bypasses role filter)
- Cosine similarity scoring
- Pass/fail table
- Histogram of scores
- Prompt improvement loop demo

In [ ]:
import sys
sys.path.insert(0, '..')

from sqlalchemy import select

from policy_system.db.session import get_session
from policy_system.db.models import User, Role, RoleType
from policy_system.llm.ollama_provider import OllamaProvider
from policy_system.rag.chromadb_provider import ChromaDBProvider
from policy_system.validation.runner import load_gold_standards, run_validation, print_validation_report
from policy_system.validation.scorer import score_answer, evaluate_batch, DEFAULT_PASS_THRESHOLD

llm = OllamaProvider()
rag = ChromaDBProvider(persist_dir='../data/chroma_db', collection_name='notebook_04_test')

print(f'ChromaDB chunks available: {rag.get_chunk_count()}')
print('Ready')

## 1. Load Gold Standards

In [ ]:
async def load_questions():
    async with get_session() as session:
        # Get or create the Global Auditor user for running QA
        auditor = (await session.execute(select(User).where(User.email == 'auditor@example.com'))).scalar_one_or_none()
        if auditor is None:
            print('No auditor user found — run notebook 01 first')
            return None, None
        
        questions = await load_gold_standards(
            session,
            created_by=auditor.user_id,
        )
        print(f'Loaded {len(questions)} canned questions:')
        for q in questions:
            print(f'  [{q.domain or "all"}] {q.question_text[:60]}...')
        return questions, auditor

questions, auditor_user = await load_questions()

## 2. Run Validation Harness

In [ ]:
if questions and auditor_user:
    async def run_all():
        async with get_session() as session:
            runs = await run_validation(
                session=session,
                rag_provider=rag,
                llm_provider=llm,
                runner_user=auditor_user,
                questions=questions,
                pass_threshold=DEFAULT_PASS_THRESHOLD,
            )
            return runs
    
    validation_runs = await run_all()
    print_validation_report(validation_runs, questions)

## 3. Score Distribution Histogram

In [ ]:
if questions and validation_runs:
    try:
        import matplotlib.pyplot as plt
        
        scores = [r.similarity_score for r in validation_runs]
        
        plt.figure(figsize=(8, 4))
        plt.hist(scores, bins=10, range=(0, 1), color='steelblue', edgecolor='white')
        plt.axvline(x=DEFAULT_PASS_THRESHOLD, color='red', linestyle='--', label=f'Pass threshold ({DEFAULT_PASS_THRESHOLD})')
        plt.xlabel('Cosine Similarity Score')
        plt.ylabel('Count')
        plt.title('Validation Score Distribution')
        plt.legend()
        plt.tight_layout()
        plt.show()
    except ImportError:
        print('matplotlib not installed — skipping histogram')
        print(f'Scores: {[round(r.similarity_score, 3) for r in validation_runs]}')

## 4. Standalone Scorer Test

In [ ]:
# Test the scorer without needing Ollama
print('Testing standalone cosine similarity scorer...')

# High similarity — same meaning, different words
score_high = score_answer(
    ai_answer='MFA is required for all employees accessing company systems.',
    gold_answer='All employees must use multi-factor authentication when accessing company systems.',
)
print(f'High similarity (MFA paraphrase): {score_high:.3f}')
assert score_high > 0.7, f'Expected > 0.7, got {score_high}'

# Low similarity — completely different topic
score_low = score_answer(
    ai_answer='The sky is blue and clouds are white.',
    gold_answer='Passwords must be changed every 90 days and be at least 12 characters.',
)
print(f'Low similarity (unrelated): {score_low:.3f}')
assert score_low < 0.5, f'Expected < 0.5, got {score_low}'

print('Scorer: PASS')

## 5. Prompt Improvement Loop Demo

In [ ]:
# Demo: compare two prompt variants for the same question
# In a real improvement loop, you'd modify prompts.py and re-run

gold = 'Passwords must be at least 12 characters and changed every 90 days.'

# Simulate two different AI responses (as if from different prompt variants)
response_v1 = 'Employee passwords should be long and changed regularly per company policy.'
response_v2 = 'The IT Security Policy requires passwords to be minimum 12 characters and rotated every 90 days.'

score_v1 = score_answer(response_v1, gold)
score_v2 = score_answer(response_v2, gold)

print('Prompt comparison:')
print(f'  Prompt v1 (vague):    score={score_v1:.3f}')
print(f'  Prompt v2 (specific): score={score_v2:.3f}')
print(f'  Winner: {"v2" if score_v2 > score_v1 else "v1"}')
print('\nUse this loop to compare prompts.py variants and pick the best one.')

## Summary

Validation harness complete:
- Gold standards loaded from YAML → canned_questions table ✓
- Validation runs stored in validation_runs table ✓
- Cosine similarity scorer (sentence-transformers) ✓
- Pass/fail table with threshold=0.75 ✓
- Score distribution histogram ✓
- Prompt comparison loop demo ✓

**System is now fully built.** See the End-to-End Verification checklist in the build plan.